In [1]:
pip install streamlit openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.1 MB/s eta 0:00:00


In [2]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import davies_bouldin_score
import warnings

warnings.filterwarnings('ignore')

st.set_page_config(layout='wide', page_title='Analisis Clustering Produk Penjualan')

st.title('📊 Analisis Clustering Data Penjualan Januari – Maret')
st.write('Aplikasi ini melakukan analisis clustering pada data penjualan produk berdasarkan Kuantitas Total dan Harga per Unit.')

# --- 1. Load Data ---
@st.cache_data
def load_data(filepath_jan, filepath_feb, filepath_mar):
    def baca_xlsx(filepath, bulan):
        df = pd.read_excel(filepath, dtype=str)
        df['Bulan'] = bulan

        df['Qty_numeric_temp'] = pd.to_numeric(df['Qty'], errors='coerce')
        df['HargaSatuan_numeric_temp'] = pd.to_numeric(df['Harga Satuan'], errors='coerce')

        category_mask = df['Qty_numeric_temp'].isna() & df['HargaSatuan_numeric_temp'].isna() & df['SKU'].notna()

        df['Kategori'] = np.nan
        df.loc[category_mask, 'Kategori'] = df.loc[category_mask, 'SKU']
        df['Kategori'] = df['Kategori'].ffill()

        df = df[df['SKU'].astype(str).str.match(r'^\d', na=False)].copy()
        df.drop(columns=['Qty_numeric_temp', 'HargaSatuan_numeric_temp'], inplace=True, errors='ignore')
        return df

    df_jan = baca_xlsx(filepath_jan, 'Januari')
    df_feb = baca_xlsx(filepath_feb, 'Februari')
    df_mar = baca_xlsx(filepath_mar, 'Maret')

    df_raw = pd.concat([df_jan, df_feb, df_mar], ignore_index=True)
    return df_raw

# NOTE: Update these file paths to where your files are located in Colab or Google Drive
# If running locally, ensure these files are in the same directory as app.py or provide full paths.
FILEPATH_JAN = '/content/drive/MyDrive/BISMILLAH KP/Ringkasan Penjualan Produk - Januari.xlsx'
FILEPATH_FEB = '/content/drive/MyDrive/BISMILLAH KP/Ringkasan Penjualan Produk - Februari.xlsx'
FILEPATH_MAR = '/content/drive/MyDrive/BISMILLAH KP/Ringkasan Penjualan Produk - Maret.xlsx'

with st.spinner('Memuat dan membersihkan data...'):
    df_raw = load_data(FILEPATH_JAN, FILEPATH_FEB, FILEPATH_MAR)
    st.success('Data berhasil dimuat!')

# --- 2. Data Cleaning ---
@st.cache_data
def clean_data(df_raw):
    df = df_raw.copy()

    kolom_numerik = ['Qty', 'Harga Satuan', 'Diskon',
                     'Harga Jual Setelah Diskon', 'Cogs', 'Gross Profit']
    for kol in kolom_numerik:
        if kol in df.columns:
            df[kol] = pd.to_numeric(df[kol], errors='coerce')

    df = df.dropna(subset=['Produk', 'Qty', 'Harga Satuan'])
    df = df[df['Qty'] > 0]
    df = df[df['Harga Satuan'] > 0]

    df['Harga_Unit'] = (df['Harga Satuan'] / df['Qty']).round(0)

    df['Diskon'] = df['Diskon'].fillna(0)
    if 'Gross Profit' in df.columns:
        df['Gross Profit'] = df['Gross Profit'].fillna(0)

    EXCL_KATEGORI = [
        'SPECIAL RAMADHAN',
        "VALENTINE'S SPECIAL PACKAGE",
        'VALENTINE PACKAGE',
        'Paket Event',
        'FREE [Deleted]',
        'Special Bundling [Deleted]',
    ]
    EXCL_PRODUK = [
        'Mineral',
        'Ice Cube',
        'Nasi Putih',
    ]

    mask_kategori = df['Kategori'].isin(EXCL_KATEGORI)
    mask_produk   = df['Produk'].isin(EXCL_PRODUK)
    mask_eksklusif = mask_kategori | mask_produk

    df_filtered = df[~mask_eksklusif].copy()
    df_filtered = df_filtered.drop_duplicates(subset=['SKU','Bulan'], keep='first')
    return df_filtered

with st.spinner('Membersihkan dan memproses data...'):
    df_filtered = clean_data(df_raw)
    st.success('Data berhasil dibersihkan!')

# --- 3. Data Processing (Aggregation) ---
@st.cache_data
def aggregate_data(df_filtered):
    def qty_bulan(grp, bulan):
        val = grp.loc[grp['Bulan'] == bulan, 'Qty']
        return val.sum() if len(val) > 0 else 0

    agg_list = []
    for (sku, produk), grp in df_filtered.groupby(['SKU','Produk']):
        qty_jan = qty_bulan(grp, 'Januari')
        qty_feb = qty_bulan(grp, 'Februari')
        qty_mar = qty_bulan(grp, 'Maret')
        total_qty     = grp['Qty'].sum()
        total_revenue = grp['Harga Satuan'].sum()
        total_profit  = grp['Gross Profit'].sum() if 'Gross Profit' in grp.columns else 0
        harga_unit    = round(total_revenue / total_qty) if total_qty > 0 else 0
        bulan_muncul  = grp['Bulan'].nunique()
        agg_list.append({
            'SKU'            : sku,
            'Produk'         : produk,
            'Qty_Jan'        : int(qty_jan),
            'Qty_Feb'        : int(qty_feb),
            'Qty_Mar'        : int(qty_mar),
            'Total_Qty'      : int(total_qty),
            'Harga_Unit'     : int(harga_unit),
            'Total_Revenue'  : int(total_revenue),
            'Total_Profit'   : round(total_profit, 0),
            'Bulan_Muncul'   : bulan_muncul,
        })

    df_agg = pd.DataFrame(agg_list).sort_values('Total_Qty', ascending=False).reset_index(drop=True)
    df_agg[['Total_Qty', 'Harga_Unit', 'Total_Revenue', 'Total_Profit']] = df_agg[['Total_Qty', 'Harga_Unit', 'Total_Revenue', 'Total_Profit']].astype(float)
    return df_agg

with st.spinner('Melakukan agregasi data...'):
    df_agg = aggregate_data(df_filtered)
    st.success('Data berhasil diagregasi!')

st.subheader('Pratinjau Data Agregasi Produk')
st.dataframe(df_agg.head())

# --- 4. Clustering ---
# Standardize the data
X = df_agg[['Total_Qty', 'Harga_Unit']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

st.sidebar.header('Pengaturan Clustering')
K_FINAL = st.sidebar.slider('Pilih jumlah cluster (K):', min_value=2, max_value=6, value=5, step=1)

# Perform KMeans clustering
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10, max_iter=300)
df_agg['Cluster_Raw'] = km_final.fit_predict(X_scaled)

# Label clusters based on mean Total_Qty
mean_qty = df_agg.groupby('Cluster_Raw')['Total_Qty'].mean().sort_values()
if K_FINAL == 3:
    label_map = {
        mean_qty.index[0]: 'Rendah',
        mean_qty.index[1]: 'Sedang',
        mean_qty.index[2]: 'Tinggi',
    }
elif K_FINAL == 2:
    label_map = {mean_qty.index[0]: 'Rendah', mean_qty.index[1]: 'Tinggi'}
else:
    label_map = {idx: f'Cluster {i+1}' for i, idx in enumerate(mean_qty.index)}

df_agg['Tingkat_Penjualan'] = df_agg['Cluster_Raw'].map(label_map)

# Centroid
centroid_asli = df_agg.groupby('Tingkat_Penjualan')[['Total_Qty', 'Harga_Unit']].mean().round(2)

st.subheader(f'Hasil Clustering (K={K_FINAL})')
st.write(f'Jumlah produk unik: {len(df_agg)}')
st.write('Distribusi produk per cluster:')
st.dataframe(df_agg['Tingkat_Penjualan'].value_counts().sort_index())

st.write('Karakteristik rata-rata cluster:')
st.dataframe(centroid_asli)

# --- 5. Visualisasi ---

st.subheader('Visualisasi Hasil Clustering')

# Define colors and markers for visualization
warna = {
    'Cluster 1': '#e74c3c',
    'Cluster 2': '#f39c12',
    'Cluster 3': '#27ae60',
    'Cluster 4': '#3498db',
    'Cluster 5': '#9b59b6',
    'Rendah': '#e74c3c',
    'Sedang': '#f39c12',
    'Tinggi': '#27ae60',
}
marker_shape = {'Rendah': 'o', 'Sedang': 's', 'Tinggi': '^'}

# Scatter Plot
fig_scatter, ax_scatter = plt.subplots(figsize=(11, 7))

# Get current cluster order for consistent plotting
if K_FINAL == 3:
    plot_order = ['Rendah', 'Sedang', 'Tinggi']
else:
    # Ensure consistent order for K>3 based on sorted mean_qty
    plot_order = [label_map[idx] for idx in mean_qty.index]

for tingkat in plot_order:
    sub = df_agg[df_agg['Tingkat_Penjualan'] == tingkat]
    ax_scatter.scatter(sub['Total_Qty'], sub['Harga_Unit'],
                       c=warna.get(tingkat, '#999'),
                       label=f'{tingkat} ({len(sub)} produk)',
                       s=80, alpha=0.80, edgecolors='white', linewidth=0.6)

    # Annotate top products to avoid clutter
    for _, row in sub.nlargest(min(3, len(sub)), 'Total_Qty').iterrows():
        ax_scatter.annotate(row['Produk'], xy=(row['Total_Qty'], row['Harga_Unit']),
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=7, alpha=0.8,
                            arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

# Centroid stars
for tingkat, row in centroid_asli.iterrows():
    ax_scatter.scatter(row['Total_Qty'], row['Harga_Unit'],
                       c='black', marker='*', s=400,
                       edgecolors='white', linewidth=0.8, zorder=10,
                       label=f'Centroid {tingkat}')

ax_scatter.set_title(f'Scatter Plot Clustering K-Means (K={K_FINAL})',
                     fontsize=13, fontweight='bold', pad=12)
ax_scatter.set_xlabel('Total Qty Terjual (3 Bulan)', fontsize=11)
ax_scatter.set_ylabel('Harga per Unit (Rp)', fontsize=11)
ax_scatter.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp {int(x):,}'))
ax_scatter.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax_scatter.grid(True, alpha=0.25, linestyle='--')
st.pyplot(fig_scatter)

# Top Products per Cluster Bar Chart
st.subheader(f'Top 20 Produk per Cluster (K={K_FINAL})')

fig_bars, axes_bars = plt.subplots(1, K_FINAL, figsize=(7 * K_FINAL, 7), sharey=False)
if K_FINAL == 1:
    axes_bars = [axes_bars]

for ax_bar, tingkat in zip(axes_bars, plot_order):
    sub = df_agg[df_agg['Tingkat_Penjualan'] == tingkat] \
              .nlargest(20, 'Total_Qty').sort_values('Total_Qty')
    color = warna.get(tingkat, '#999')

    bars = ax_bar.barh(sub['Produk'], sub['Total_Qty'],
                       color=color, alpha=0.85, edgecolor='white', linewidth=0.5)

    for bar, val in zip(bars, sub['Total_Qty']):
        ax_bar.text(bar.get_width() + sub['Total_Qty'].max() * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f'{int(val):,}', va='center', ha='left', fontsize=8, fontweight='bold')

    ax_bar.set_title(f'Cluster {tingkat}\n({len(df_agg[df_agg["Tingkat_Penjualan"]==tingkat])} produk)',
                     fontsize=11, fontweight='bold', color=color)
    ax_bar.set_xlabel('Total Qty Terjual (Jan–Mar)', fontsize=9)
    ax_bar.tick_params(axis='y', labelsize=8)
    ax_bar.set_xlim(0, sub['Total_Qty'].max() * 1.18)
    ax_bar.grid(axis='x', alpha=0.3, linestyle='--')
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)

fig_bars.suptitle(f'Top 20 Produk per Cluster (K={K_FINAL})\nTotal Qty Terjual Gabungan Jan–Mar',
                 fontsize=13, fontweight='bold', y=1.01)
st.pyplot(fig_bars)

# Monthly Trend Plot
st.subheader('Tren Qty Total per Bulan per Cluster')
fig_tren, axes_tren = plt.subplots(1, 2, figsize=(18, 6))

tren = df_agg.groupby('Tingkat_Penjualan')[['Qty_Jan', 'Qty_Feb', 'Qty_Mar']].sum()

# Line chart
ax1_tren = axes_tren[0]
for tingkat in tren.index:
    ax1_tren.plot(['Januari', 'Februari', 'Maret'], tren.loc[tingkat],
                  marker='o', linewidth=2.2, markersize=8,
                  label=f'{tingkat}', color=warna.get(tingkat, '#999'))
    for i, (bulan, val) in enumerate(tren.loc[tingkat].items()):
        ax1_tren.annotate(f'{int(val):,}',
                          xy=(i, val), xytext=(0, 10),
                          textcoords='offset points', ha='center', fontsize=9)

ax1_tren.set_title('Tren Total Qty Terjual per Cluster\n(Januari – Maret)', fontsize=11, fontweight='bold')
ax1_tren.set_ylabel('Total Qty', fontsize=10)
ax1_tren.legend(fontsize=9)
ax1_tren.grid(True, alpha=0.3, linestyle='--')
ax1_tren.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Stacked bar
ax2_tren = axes_tren[1]
x_tren = np.arange(3)
bulan_label = ['Januari', 'Februari', 'Maret']
bottom_tren = np.zeros(3)
for tingkat in tren.index:
    vals = tren.loc[tingkat].values
    ax2_tren.bar(x_tren, vals, bottom=bottom_tren, label=f'{tingkat}',
                 color=warna.get(tingkat, '#999'), alpha=0.85, edgecolor='white')
    for xi, (v, b) in enumerate(zip(vals, bottom_tren)):
        if v > 0:
            ax2_tren.text(xi, b + v/2, f'{int(v):,}',
                          ha='center', va='center', fontsize=8.5,
                          fontweight='bold', color='white')
    bottom_tren += vals

ax2_tren.set_xticks(x_tren)
ax2_tren.set_xticklabels(bulan_label)
ax2_tren.set_title('Komposisi Penjualan per Cluster per Bulan', fontsize=11, fontweight='bold')
ax2_tren.set_ylabel('Total Qty', fontsize=10)
ax2_tren.legend(fontsize=9)
ax2_tren.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2_tren.grid(axis='y', alpha=0.3, linestyle='--')
st.pyplot(fig_tren)

st.subheader('Distribusi Jumlah Produk & Sebaran Harga per Unit per Cluster')
fig_dist_price, axes_dist_price = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
ax1_dist = axes_dist_price[0]
cluster_order_pie = plot_order
sizes  = [len(df_agg[df_agg['Tingkat_Penjualan']==t]) for t in cluster_order_pie]
colors_pie = [warna.get(t,'#999') for t in cluster_order_pie]
explode = [0.03] * len(cluster_order_pie)

wedges, texts, autotexts = ax1_dist.pie(
    sizes, labels=cluster_order_pie, colors=colors_pie,
    autopct=lambda p: f'{p:.1f}%\n({int(round(p*sum(sizes)/100))} produk)',
    startangle=140, explode=explode,
    textprops={'fontsize': 10})
for at in autotexts:
    at.set_fontsize(9)
ax1_dist.set_title('Distribusi Jumlah Produk per Cluster', fontsize=11, fontweight='bold')

# Box plot
ax2_price = axes_dist_price[1]
data_box  = [df_agg[df_agg['Tingkat_Penjualan']==t]['Harga_Unit'].values
             for t in cluster_order_pie]
bp = ax2_price.boxplot(data_box, labels=cluster_order_pie,
                 patch_artist=True, notch=False,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_pie):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax2_price.set_title('Sebaran Harga per Unit per Cluster', fontsize=11, fontweight='bold')
ax2_price.set_ylabel('Harga per Unit (Rp)', fontsize=10)
ax2_price.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp {int(x):,}'))
ax2_price.grid(axis='y', alpha=0.3, linestyle='--')
st.pyplot(fig_dist_price)

st.subheader('Rekomendasi Berdasarkan Clustering')

total_qty_semua = df_agg['Total_Qty'].sum()

for tingkat in plot_order:
    sub = df_agg[df_agg['Tingkat_Penjualan'] == tingkat]
    if not sub.empty: # Check if cluster is not empty
        pct_produk = len(sub) / len(df_agg) * 100
        pct_qty    = sub['Total_Qty'].sum() / total_qty_semua * 100
        top3       = sub.nlargest(min(3, len(sub)),'Total_Qty')['Produk'].tolist()

        st.markdown(f"""
        ---
        ### Cluster {tingkat.upper()}
        - Jumlah produk  : {len(sub):>4} ({pct_produk:.1f}% dari total produk)
        - Total qty      : {int(sub['Total_Qty'].sum()):>8,} ({pct_qty:.1f}% dari total penjualan)
        - Rata-rata qty  : {sub['Total_Qty'].mean():>8.1f}
        - Rata-rata harga: Rp {sub['Harga_Unit'].mean():>10,.0f}
        - Produk terlaris: {', '.join(top3)}
        """)

if K_FINAL == 3:
    st.markdown("""
    ### Rekomendasi Umum (K=3):
    1.  **Cluster TINGGI**: Prioritaskan stok & promosi produk unggulan.
    2.  **Cluster SEDANG**: Optimalkan strategi bundling/upselling.
    3.  **Cluster RENDAH**: Evaluasi produk dengan penjualan terendah untuk perubahan strategi atau penghentian.
    """)
elif K_FINAL > 3:
    st.markdown("""
    ### Rekomendasi Umum (K>3):
    Rekomendasi lebih spesifik dapat dirumuskan dengan menganalisis karakteristik masing-masing cluster secara detail, seperti:
    - **Produk volume tinggi, harga menengah**: Jaga ketersediaan, promo massal.
    - **Produk harga tinggi, volume menengah**: Pertahankan kualitas, targetkan segmen premium.
    - **Produk volume rendah**: Pertimbangkan promosi, diskon, atau evaluasi ulang keberadaannya.
    """)

st.info('Untuk melihat detail produk di setiap cluster, gunakan data yang diekspor ke Excel.')


Writing app.py
